# De Laval nozzle example with Gaslab

In [1]:
import math
import numpy as np
import matplotlib.pyplot as plt
from scipy.optimize import root_scalar

from gaslab import GasState
from gaslab.relations import arstar, oblique


© Tim Colonius, California Institute of Technology

Revised: March 12, 2026

Converging-diverging nozzle

A converging-diverging nozzle can be ananlyzed using the Gaslab routines.  A textbook should be consulted for a complete description of the regimes of operation of such a nozzle.  The key points are summarized here.  The key design feature of the nozzle is the ratio of the exit area to the throat area, $A_e/A_t$. There are 4 regimes of operation that depend on the ratio of the ambient pressure to which the nozzle exhausts to which the stagnation pressure at the inlet, p_a / p_0.  As this ratio is decreased from unity (either by reducing ambient pressure or increasing the stagnation pressure, the regimes are

- Subsonic flow throughout.  The flow is not choked at the throat.  The pressure at the nozzle exit is matched to ambient pressure, and this condition determines, for a fixed A_e/A_t, the mass flow rate in the nozzle.  The mass flow rate starts at zero when p_a / p_0 = 1, and increases as p_a / p_0 is reduced.  The mass flow rate reaches a maximal value {\dot m}_{1-2} as the pressure ratio is reduced to a critical value, p_a / p_0 = \pi_{1-2}that occurs as the flow just chokes at the throat (the Mach number is just below unity at this point).

- Mixed flow with a normal shock at some position in the diverging section.  As the pressure ratio is reduced below \pi_{1-2}, the flow passes through the sonic point at the throat and is supersonic in the portion or diverging section upstream of the shock.  After the shock, the flow is, of course, subsonic, and it then decelerates in the remaining diverging section.   The pressure of the subsonic flow at the nozzle exit must be equal to ambient pressure, and this condition determines the position of the shock within the diverging section.  As the pressure ratio is further reduced to p_a / p_0 = \pi_{2-3}, the shock just reaches the exit plane of the nozzle.

- Supersonic flow, "overexpanded" jet at exit.  As the pressure ratio is just reduced below p_a / p_0 = \pi_{2-3}, the normal shock moves out of the exit and the flow is compressed to ambient pressure through a system of oblique shocks waves external to the nozzle.  As the pressure ratio is further reduced, it eventually comes to the value where p_a=p_e, and at this point the flow is called "perfectly expanded".  We will label this critical pressure p_a / p_0 = \pi_{3-4}

- Supersonic flow, "underexanded" jet after exit.  Upon further reduction of p_a / p_0 , the nozzle remains in supersonic flow but the flow is exiting at a pressure where p_a < p_e, and the flow must continue to expand towards atmospheric pressure outside the nozzle (through a system of Prandtl-meyer expansions).

Example 1.  For air and A_e/A_t = 5, determine the values of \pi_{1-2}, \pi_{2-3} and \pi_{3-4}.  

The easiest place to start is \pi_{1-2}, as this corresponds to isentropic acceleration of subsonic flow to being just choked at the throat.  Then at the nozzle exit, we have A_e/A_t = A_e / A^*, so that we can find the exit Mach number as

In [2]:
zerofun = lambda me: 5 - GasState.init(me, 1.4).arcrit  # define function handle to find mexit
m12 = root_scalar(zerofun, bracket=[0.001, 1], method="brentq").root  # subsonic solution


Remembering that Gaslab (in dimensionless mode) returns the pressure relative to the stagnation pressure of the intiialized state, and that the flow throughout the nozzle is isentropic, we have, simply

In [3]:
pi12 = GasState.init(m12, 1.4).pres


It is easiest to skip now to \pi_{3-4}, where we have instead isentropic acceleration from the stagnation pressure to a supersonic flow just at ambient pressure.  Once again A_e/A_t = A_e / A^*, but we want the supersonic branch of the solution

In [4]:
m34 = root_scalar(zerofun, bracket=[1, 20], method="brentq").root  # supersonic solution


Remembering that Gaslab (in dimensionless mode) returns the pressure relative to the stagnation pressure of the intiialized state, and that the flow throughout the nozzle is isentropic, we have, simply

In [5]:
pi34 = GasState.init(m34, 1.4).pres


Now we can find \pi_{2-3} by simply appending a normal shock at the exit plane:

In [6]:
pi23 = GasState.init(m34, 1.4).normalshock().pres


Example 2.  For air and A_e/A_t = 5, determine where the shock occurs in the diverging section if \pi_{1-2} > p_a / p_0 = 0.5 > \pi_{2-3}.

This is a nozzle in regime 2, and asking 'where' the shock occurs is a slightly funny question because we have not acutally specified anything about the nozzle geometry beyond A_e/A_t!  What we are really asking is what is the area (relative to the throat) where this shock wave occurs.  Let's call this A_s / A_t.  One thing we know for sure is that the Mach number just upstream of the shock 1 < M_{s1} < M_{3-4} = 3.17.  For M_{s1} =1, we would have p_a / p_0 = \pi_{1-2} = 0.99 and for M_{s1} =3.17, we would have p_a / p_0 = \pi_{2-3}=0.24.   We can set up a function handle to return the pressure ratio at the exit of the nozzle as a function of A_s / A_t (which we will denote as "asr").  We further note that {A_e \over A_t} = {A_e \over A_s} {A_s \over a_t}.

In [7]:
prexit = lambda asr: GasState.init(1, 1.4).areachg(asr, throat=True).normalshock().areachg(5 / asr).pres


Note that in the area change from throat to shock, we used the optional 'throat=true' argumet to insist on getting the supersonic solution from the choke point (type "help areachg" for details).  Now we can find the asr to match prexit = 0.5:

In [8]:
zerofun = lambda asr: 0.5 - prexit(asr)
asr = root_scalar(zerofun, x0=2.0, x1=3.0, method="secant").root


In [9]:
print(asr)

2.5197074227475613


Ok, that was painless!  We found that when p_a / p_0 = 0.5 then a shock must sit at a position where the {A_s/A_t}=2.5.

In [10]:
help(GasState)

Help on class GasState in module gaslab.state:

class GasState(builtins.object)
 |  GasState(m: float, gamma: float, p: Optional[float] = None, T: Optional[float] = None, mw: Optional[float] = None, p0: float = 1.0, t0: float = 1.0, angle: float = 0.0, proc: str = 'initstate', dimmode: bool = False, gcon: float = 1.0, p0r: float = 1.0, t0r: float = 1.0, hist: Optional[ForwardRef('GasState')] = None, defaults: gaslab.defaults.Defaults = <factory>) -> None
 |  
 |  Gas dynamic flow state used in gaslab.
 |  
 |  Methods defined here:
 |  
 |  __eq__(self, other)
 |      Return self==value.
 |  
 |  __init__(self, m: float, gamma: float, p: Optional[float] = None, T: Optional[float] = None, mw: Optional[float] = None, p0: float = 1.0, t0: float = 1.0, angle: float = 0.0, proc: str = 'initstate', dimmode: bool = False, gcon: float = 1.0, p0r: float = 1.0, t0r: float = 1.0, hist: Optional[ForwardRef('GasState')] = None, defaults: gaslab.defaults.Defaults = <factory>) -> None
 |      Initial